# TP Data Integration & Applications — ST2DLDI
**Dataset :** Accidents corporels de la circulation routière — France 2024 (data.gouv.fr)  
**Architecture :** Medallion (Bronze → Silver → Gold)

## Partie 1 — Data Profiling & Data Quality

### Chargement des données (couche Bronze)

In [1]:
import pandas as pd
import numpy as np

caract    = pd.read_csv('data/caract-2024.csv',    sep=';', encoding='latin1', low_memory=False)
lieux     = pd.read_csv('data/lieux-2024.csv',     sep=';', encoding='latin1', low_memory=False)
usagers   = pd.read_csv('data/usagers-2024.csv',   sep=';', encoding='latin1', low_memory=False)
vehicules = pd.read_csv('data/vehicules-2024.csv',  sep=';', encoding='latin1', low_memory=False)

tables = {
    'caracteristiques': caract,
    'lieux':            lieux,
    'usagers':          usagers,
    'vehicules':        vehicules,
}

for name, df in tables.items():
    print(f'  {name:<20} {len(df):>7} lignes  {df.shape[1]} colonnes')

  caracteristiques       54402 lignes  15 colonnes
  lieux                  70248 lignes  18 colonnes
  usagers               125187 lignes  16 colonnes
  vehicules              92678 lignes  11 colonnes


### Exploration rapide des données

In [2]:
#premières lignes de chaque table
caract.head(3)

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long
0,202400000001,25,3,2024,07:40,2,70,70285,1,1,5,1,D438,"47,56277000","6,75832000"
1,202400000002,20,3,2024,15:05,1,21,21054,2,3,7,6,HOTEL DIEU (RUE DE L'),"47,02109000","4,83755000"
2,202400000003,22,3,2024,19:30,2,15,15012,1,1,1,6,AllÃ©e des Tilleuls,"44,90238400","2,49641800"


In [3]:
lieux.head(3)

,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202400000001,3,D438,0,NaN,2,2,0,1,1,260,2,NaN,7,1,0,1,90
1,202400000002,4,HOTEL DIEU (RUE DE L'),0,NaN,2,2,0,1,-1,-1,1,NaN,-1,9,0,1,30
2,202400000002,4,POTERNE (RUE),0,NaN,1,1,0,1,-1,-1,1,NaN,-1,9,0,1,30


In [4]:
usagers.head(3)

,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202400000001,203Â 988Â 581,155Â 781Â 758,A01,1,1,3,1,2003.0,2,1,-1,-1,-1,-1,-1
1,202400000001,203Â 988Â 582,155Â 781Â 759,B01,1,1,1,1,1997.0,4,1,-1,-1,-1,-1,-1
2,202400000002,203Â 988Â 579,155Â 781Â 757,A01,10,3,3,2,1927.0,5,0,-1,-1,3,3,1


In [5]:
vehicules.head(3)

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202400000001,155Â 781Â 758,A01,1,7,0,2,1,13,1,NaN
1,202400000001,155Â 781Â 759,B01,2,14,0,2,2,21,1,NaN
2,202400000002,155Â 781Â 757,A01,1,10,0,1,3,15,1,NaN


In [6]:
#statistiques descriptives
caract.describe()

,Num_Acc,jour,mois,an,lum,agg,int,atm,col
count,5.440200e+04,54402.000000,54402.000000,54402.0,54402.000000,54402.000000,54402.000000,54402.000000,54402.000000
mean,2.024000e+11,15.665490,6.615749,2024.0,1.949708,1.625161,2.108195,1.677493,4.018069
std,1.570465e+04,8.737761,3.361701,0.0,1.488462,0.484086,2.042602,1.736956,1.984560
min,2.024000e+11,1.000000,1.000000,2024.0,1.000000,1.000000,1.000000,1.000000,-1.000000
25%,2.024000e+11,8.000000,4.000000,2024.0,1.000000,1.000000,1.000000,1.000000,3.000000
50%,2.024000e+11,16.000000,7.000000,2024.0,1.000000,2.000000,1.000000,1.000000,3.000000
75%,2.024000e+11,23.000000,10.000000,2024.0,3.000000,2.000000,2.000000,1.000000,6.000000
max,2.024001e+11,31.000000,12.000000,2024.0,5.000000,2.000000,9.000000,9.000000,7.000000


In [7]:
usagers.describe()

,Num_Acc,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,etatp
count,1.251870e+05,125187.000000,125187.000000,125187.000000,125187.000000,122608.000000,125187.000000,125187.000000,125187.000000,125187.000000,125187.000000,125187.000000
mean,2.024000e+11,2.100745,1.335554,2.524208,1.272696,1985.075411,3.069009,1.938788,0.843906,-0.800994,-0.223002,-0.814605
std,1.567431e+04,2.584540,0.610862,1.374424,0.559575,19.327486,2.762129,2.384028,2.910122,1.067398,1.276891,0.638087
min,2.024000e+11,-1.000000,1.000000,1.000000,-1.000000,1914.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,2.024000e+11,1.000000,1.000000,1.000000,1.000000,1971.000000,0.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000
50%,2.024000e+11,1.000000,1.000000,3.000000,1.000000,1988.000000,4.000000,1.000000,0.000000,-1.000000,0.000000,-1.000000
75%,2.024000e+11,2.000000,2.000000,4.000000,2.000000,2001.000000,5.000000,2.000000,0.000000,-1.000000,0.000000,-1.000000
max,2.024001e+11,10.000000,3.000000,4.000000,2.000000,2024.000000,9.000000,9.000000,9.000000,9.000000,9.000000,3.000000


### A. Structure des tables

In [8]:
for name, df in tables.items():
    print(f'--- {name.upper()} ---')
    for col, dtype in df.dtypes.items():
        print(f'  {col:<30} {str(dtype)}')
    print()

--- CARACTERISTIQUES ---
  Num_Acc                        int64
  jour                           int64
  mois                           int64
  an                             int64
  hrmn                           object
  lum                            int64
  dep                            object
  com                            object
  agg                            int64
  int                            int64
  atm                            int64
  col                            int64
  adr                            object
  lat                            object
  long                           object

--- LIEUX ---
  Num_Acc                        int64
  catr                           int64
  voie                           object
  v1                             int64
  v2                             object
  circ                           int64
  nbv                            object
  vosp                           int64
  prof                           int64
  pr           

### A. Semantic meaning — signification des colonnes

**Source :** dictionnaire de donnees officiel BAAC (Bulletin d'Analyse des Accidents Corporels), publie par l'ONISR sur data.gouv.fr aux cotes des fichiers CSV (rubrique *Description des bases de donnees annuelles*). Liste non exhaustive ci-dessous : quelques colonnes cles par table pour montrer la comprehension du schema, le detail complet des codes est dans le dictionnaire cite en source.

**Table caracteristiques** (1 ligne = 1 accident)

| Colonne | Signification |
|---|---|
| Num_Acc | Identifiant unique de l'accident |
| lum | Conditions d'eclairage (1=plein jour, 2=crepuscule, 3=nuit sans eclairage, ...) |
| atm | Conditions meteorologiques (1=normale, 2=pluie, 3=neige, ...) |
| col | Type de collision (1=frontal, 2=arriere, 3=cote, ...) |

**Table lieux** (caracteristiques de la route)

| Colonne | Signification |
|---|---|
| catr | Categorie de route (1=autoroute, 2=nationale, 3=departementale, ...) |
| surf | Etat de la surface (1=normale, 2=mouillee, 3=flaques, ...) |
| vma | Vitesse maximale autorisee |

**Table usagers** (1 ligne = 1 personne impliquee)

| Colonne | Signification |
|---|---|
| catu | Categorie d'usager (1=conducteur, 2=passager, 3=pieton) |
| grav | Gravite (1=indemne, 2=blesse hospitalise, 3=blesse leger, 4=tue) |
| secu1 | Equipement de securite (1=ceinture, 2=casque, 3=gilet, ...) |

**Table vehicules** (1 ligne = 1 vehicule implique)

| Colonne | Signification |
|---|---|
| catv | Categorie de vehicule (1=bicyclette, 2=cyclomoteur, 7=voiture, ...) |
| choc | Point de choc initial (1=avant, 2=avant droit, 3=avant gauche, ...) |
| motor | Type de motorisation (1=hydrocarbures, 2=hybride, 3=electrique, ...) |

### B. Valeurs manquantes

In [9]:
for name, df in tables.items():
    print(f'--- {name.upper()} ---')
    missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    missing = missing[missing > 0]
    if missing.empty:
        print('  aucune valeur manquante')
    else:
        for col, pct in missing.items():
            flag = ' CRITIQUE' if pct > 20 else ''
            print(f'  {col:<30} {pct:.1f}%{flag}')
    print()

--- CARACTERISTIQUES ---
  adr                            4.2%

--- LIEUX ---
  lartpc                         100.0% CRITIQUE
  v2                             91.6% CRITIQUE
  voie                           19.0%

--- USAGERS ---
  an_nais                        2.1%

--- VEHICULES ---
  occutc                         99.0% CRITIQUE



### C. Cohérence et validité

In [10]:
#coordonnees GPS
caract['lat_f']  = pd.to_numeric(caract['lat'].str.replace(',', '.'),  errors='coerce')
caract['long_f'] = pd.to_numeric(caract['long'].str.replace(',', '.'), errors='coerce')

hors_metro = caract[(caract['lat_f'] < 41) | (caract['lat_f'] > 52)].shape[0]
print(f'Coordonnees hors France metro (DOM-TOM) : {hors_metro}')

#valeurs negatives
for col in ['grav', 'sexe', 'place']:
    neg = (usagers[col] < 0).sum()
    if neg > 0:
        print(f'usagers.{col} : {neg} valeurs negatives (-1 = non renseigne)')

#ages (derives depuis an_nais)
age = 2024 - usagers['an_nais']
ages_negatifs = (age < 0).sum()
ages_aberrants = (age > 110).sum()
print(f'Ages negatifs : {ages_negatifs}')
print(f'Ages > 110 ans (aberrants) : {ages_aberrants}')

#doublons
print()
for name, df in tables.items():
    dup = df.duplicated().sum()
    print(f'  {name:<20} {dup} doublon(s)')

Coordonnees hors France metro (DOM-TOM) : 3339
usagers.sexe : 2395 valeurs negatives (-1 = non renseigne)
usagers.place : 3 valeurs negatives (-1 = non renseigne)
Ages negatifs : 0
Ages > 110 ans (aberrants) : 0

  caracteristiques     0 doublon(s)
  lieux                2 doublon(s)


  usagers              0 doublon(s)


  vehicules            0 doublon(s)


### C. Anomalies catégorielles

In [11]:
#valeurs attendues par colonne categorielle
categories_attendues = {
    'grav':   [1, 2, 3, 4],
    'sexe':   [1, 2],
    'catu':   [1, 2, 3, 4],
    'trajet': [0, 1, 2, 3, 4, 5, 9],
}

for col, valides in categories_attendues.items():
    valeurs_uniques = usagers[col].dropna().unique()
    inattendues = [v for v in valeurs_uniques if v not in valides]
    if inattendues:
        print(f'usagers.{col} : valeurs inattendues -> {inattendues}')
    else:
        print(f'usagers.{col} : OK (valeurs = {sorted(valeurs_uniques.tolist())})')

print()
#verification colonne lum et atm dans caract
for col, valides in {'lum': [1,2,3,4,5], 'atm': [1,2,3,4,5,6,7,8,9]}.items():
    vals = caract[col].dropna().unique()
    inattendues = [v for v in vals if v not in valides]
    if inattendues:
        print(f'caract.{col} : valeurs inattendues -> {inattendues}')
    else:
        print(f'caract.{col} : OK')

usagers.grav : OK (valeurs = [1, 2, 3, 4])
usagers.sexe : valeurs inattendues -> [np.int64(-1)]
usagers.catu : OK (valeurs = [1, 2, 3])
usagers.trajet : valeurs inattendues -> [np.int64(-1)]

caract.lum : OK
caract.atm : OK


### D. Résumé qualité

| Problème | Colonne | Action |
|---|---|---|
| 100% vide | lieux.lartpc | Supprimer |
| 99% vide | vehicules.occutc | Supprimer |
| 91.6% vide | lieux.v2 | Supprimer |
| 19% vide | lieux.voie | Imputer 'Inconnu' |
| 4.2% vide | caract.adr | Imputer 'Inconnu' |
| Hors métropole | lat/long | Séparer DOM-TOM |
| Valeurs -1 | sexe, place, col | Remplacer par NaN |
| Doublons | lieux | Supprimer (2 lignes) |

### D. Impact analysis

- Colonnes vides : inutilisables, a exclure avant tout traitement
- Valeurs -1 dans `grav` et `sexe` : faussent les moyennes et distributions
- Coordonnees hors metro : biaiseraient toute analyse geographique ou cartographique
- Doublons dans `lieux` : gonfleraient les comptages sur certaines routes

## Partie 2A — Transformations (couche Silver)

In [12]:
#1. suppression colonnes trop vides
lieux     = lieux.drop(columns=['lartpc', 'v2'])
vehicules = vehicules.drop(columns=['occutc'])

#2. doublons
lieux = lieux.drop_duplicates()

#3. conversion coordonnées GPS
caract['lat']  = pd.to_numeric(caract['lat'].str.replace(',', '.'),  errors='coerce')
caract['long'] = pd.to_numeric(caract['long'].str.replace(',', '.'), errors='coerce')

#4. séparation métropole / DOM-TOM
dom_tom      = caract[(caract['lat'] < 41) | (caract['lat'] > 52)]
caract_metro = caract[(caract['lat'] >= 41) & (caract['lat'] <= 52)].copy()
print(f'DOM-TOM mis de cote : {len(dom_tom)} lignes')
print(f'Metropole conservee : {len(caract_metro)} lignes')

#5. remplacement des -1 par NaN
for col in ['sexe', 'place', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'etatp']:
    usagers[col] = usagers[col].replace(-1, np.nan)
caract_metro['col'] = caract_metro['col'].replace(-1, np.nan)

#6. imputation valeurs manquantes texte
caract_metro['adr'] = caract_metro['adr'].fillna('Inconnu')
lieux['voie']       = lieux['voie'].fillna('Inconnu')

#7. colonne date
caract_metro['date'] = pd.to_datetime(
    caract_metro[['an', 'mois', 'jour']].rename(columns={'an': 'year', 'mois': 'month', 'jour': 'day'}),
    errors='coerce'
)

#8. enrichissement tranche horaire
caract_metro['heure'] = caract_metro['hrmn'].str[:2].astype(float, errors='ignore')

def categorie_horaire(h):
    if pd.isna(h):   return 'Inconnu'
    if 6  <= h < 12: return 'Matin'
    if 12 <= h < 18: return 'Apres-midi'
    if 18 <= h < 22: return 'Soir'
    return 'Nuit'

caract_metro['tranche_horaire'] = caract_metro['heure'].apply(categorie_horaire)

#9. enrichissement gravite max par accident
gravite_max = usagers.groupby('Num_Acc')['grav'].max().reset_index()
gravite_max.columns = ['Num_Acc', 'gravite_max']
caract_metro = caract_metro.merge(gravite_max, on='Num_Acc', how='left')

#sauvegarde Silver
caract_metro.to_csv('data/silver_caract.csv',    index=False)
lieux.to_csv(        'data/silver_lieux.csv',     index=False)
usagers.to_csv(      'data/silver_usagers.csv',   index=False)
vehicules.to_csv(    'data/silver_vehicules.csv', index=False)
print('[OK] fichiers Silver sauvegardes')

DOM-TOM mis de cote : 3339 lignes
Metropole conservee : 51063 lignes


[OK] fichiers Silver sauvegardes


## Partie 2B — Modélisation (couche Gold)

In [13]:
caract_s    = pd.read_csv('data/silver_caract.csv',    low_memory=False)
lieux_s     = pd.read_csv('data/silver_lieux.csv',     low_memory=False)
usagers_s   = pd.read_csv('data/silver_usagers.csv',   low_memory=False)
vehicules_s = pd.read_csv('data/silver_vehicules.csv', low_memory=False)

#table de faits
fait_accidents = caract_s[[
    'Num_Acc', 'date', 'tranche_horaire', 'gravite_max',
    'atm', 'col', 'lum', 'agg', 'dep'
]].rename(columns={
    'atm': 'meteo', 'col': 'type_collision',
    'lum': 'eclairage', 'agg': 'en_agglomeration', 'dep': 'departement'
})

#dimension lieu
dim_lieu = lieux_s[[
    'Num_Acc', 'catr', 'surf', 'infra', 'situ', 'vma', 'prof', 'plan'
]].rename(columns={
    'catr': 'type_route', 'surf': 'etat_surface', 'infra': 'infrastructure',
    'situ': 'situation', 'vma': 'vitesse_max', 'prof': 'profil_route', 'plan': 'trace_plan'
})

#dimension vehicule
dim_vehicule = vehicules_s[[
    'Num_Acc', 'id_vehicule', 'catv', 'manv', 'obs', 'obsm', 'choc', 'motor'
]].rename(columns={
    'catv': 'type_vehicule', 'manv': 'manoeuvre', 'obs': 'obstacle_fixe',
    'obsm': 'obstacle_mobile', 'choc': 'point_choc', 'motor': 'motorisation'
})

#dimension usager
dim_usager = usagers_s[[
    'Num_Acc', 'id_usager', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1'
]].rename(columns={
    'catu': 'categorie_usager', 'grav': 'gravite', 'an_nais': 'annee_naissance',
    'trajet': 'motif_trajet', 'secu1': 'equipement_securite'
})

#sauvegarde Gold
fait_accidents.to_csv('data/gold_fait_accidents.csv', index=False)
dim_lieu.to_csv(      'data/gold_dim_lieu.csv',       index=False)
dim_vehicule.to_csv(  'data/gold_dim_vehicule.csv',   index=False)
dim_usager.to_csv(    'data/gold_dim_usager.csv',     index=False)

print(f'fait_accidents : {len(fait_accidents)} lignes')
print(f'dim_lieu       : {len(dim_lieu)} lignes')
print(f'dim_vehicule   : {len(dim_vehicule)} lignes')
print(f'dim_usager     : {len(dim_usager)} lignes')
print('[OK] fichiers Gold sauvegardes')

fait_accidents : 51063 lignes
dim_lieu       : 70245 lignes
dim_vehicule   : 92678 lignes
dim_usager     : 125187 lignes
[OK] fichiers Gold sauvegardes


## Partie 2C — Diagramme Medallion Architecture

![Medallion Architecture](medallion_drawio.png)

## Justification des choix de conception

### 1. Suppression des colonnes quasi vides
Les colonnes `lartpc` (100% vide), `v2` (91.6% vide) et `occutc` (99% vide) ont ete supprimees. Une colonne avec presque aucune donnee n'apporte aucune information exploitable.

### 2. Traitement des DOM-TOM
Les 3 339 accidents hors bornes metropolitaines correspondent aux territoires d'outre-mer. Ce ne sont pas des erreurs. Nous les avons mis de cote pour limiter l'analyse a la France metropolitaine, tout en les conservant pour une etude ulterieure.

### 3. Remplacement des -1 par NaN
Dans ce dataset, -1 signifie 'non renseigne'. Le laisser tel quel fausserait les calculs statistiques. NaN est la convention standard pour une valeur inconnue en pandas.

### 4. Imputation 'Inconnu' sur les champs texte
Plutot que supprimer une ligne entiere a cause d'une adresse manquante, on conserve l'accident et on impute 'Inconnu'. Les autres informations (date, gravite, meteo) restent exploitables.

### 5. Enrichissement
- `tranche_horaire` : regroupe l'heure en 4 categories (Matin/Apres-midi/Soir/Nuit).
- `gravite_max` : gravite la plus severe parmi tous les usagers d'un accident.

### 6. Choix du modele en etoile (Star Schema)
Trois modeles etaient envisageables :
- **Table plate** : tout dans une seule table, doublons massifs, inexploitable
- **Modele en flocon (Snowflake)** : plus normalise mais complexe, justifie pour de tres grands datasets
- **Modele en etoile (Star Schema)** : choix retenu

Le modele en etoile est adapte car un accident implique plusieurs vehicules et plusieurs usagers. Il est performant pour les analyses BI, correspond a la suggestion du TP ('fact table + dimensions'), et reste simple a maintenir. La cle de jointure `Num_Acc` relie toutes les tables.